# 06 · Information Theory

**The big idea:** how do we measure *surprise*, and how wrong a probability guess is? That's exactly what a classification or language-model loss needs.

**Where this leads →** cross-entropy loss (07), the LM training objective and perplexity (10).

> Logs below use base 2 (bits) for friendliness. ML code uses natural log (nats); the only difference is a constant factor.

---

## 📂 Notes in this chapter

One short note per concept, grouped into folders. Read in order. Code builds on earlier notes, so `notebook.ipynb` (all code, in order) is the easiest way to run everything.

### `1-surprise-and-entropy/`

- [1. Information (Surprise)](1-surprise-and-entropy/01-information-surprise.md)
- [2. Entropy](1-surprise-and-entropy/02-entropy.md)

### `2-comparing-distributions/`

- [3. Cross-Entropy](2-comparing-distributions/03-cross-entropy.md)
- [4. KL Divergence](2-comparing-distributions/04-kl-divergence.md)

### `3-mutual-info-and-why/`

- [5. Mutual Information](3-mutual-info-and-why/05-mutual-information.md)
- [6. Why This Shows Up in ML and LLMs](3-mutual-info-and-why/06-why-this-shows-up-in-ml-and-llms.md)


# Information (Surprise)

*📓 Notes · [06 · Information Theory](../README.md) · surprise and entropy*

**📝 In simple words:** How surprising an event is. Rare events carry more information.

**💡 Think of it like this:** "The sun rose today" tells you nothing. "It snowed in the desert" tells you a lot.

**📐 Formula to remember:** `I(x) = −log₂ P(x)`

**✏️ Let me calculate it:** `P = 1/2` → 1 bit. `P = 1/8` → `−log₂(1/8) = 3` bits. `P = 1` → 0 bits.

**🤖 Why AI uses it:** the loss for one prediction is the surprise the model felt at the true answer: `−log P(correct)`.

In [ ]:
import numpy as np
for p in (1, 0.5, 0.125): print(p, -np.log2(p))

**🧪 Try it myself:** Information in an event with `P = 1/16`? <details><summary>Solution</summary>4 bits.</details>

---
[Entropy](../1-surprise-and-entropy/02-entropy.md) →


# Entropy

*📓 Notes · [06 · Information Theory](../README.md) · surprise and entropy*

**📝 In simple words:** The *average* surprise of a distribution, i.e. its uncertainty.

**📐 Formula to remember:** `H(p) = −Σ pᵢ log₂ pᵢ`

**🔢 Small example:** fair coin vs biased coin (0.9 / 0.1).

**✏️ Let me calculate it:**

```text
fair:   −(0.5·log₂0.5 + 0.5·log₂0.5) = 1 bit
biased: −(0.9·log₂0.9 + 0.1·log₂0.1) = 0.137 + 0.332 = 0.469 bits
```

Fair = most unpredictable. Certain outcome = 0 bits.

**🤖 Why AI uses it:** entropy of a model's output tells how unsure it is; it's the lower bound on how well any model can predict.

In [ ]:
H = lambda p: -np.sum(p * np.log2(p))
print(H(np.array([0.5, 0.5])), H(np.array([0.9, 0.1])))   # 1.0  0.469

**🧪 Try it myself:** Entropy of a fair 4-sided die? <details><summary>Solution</summary>`−4·(¼·log₂¼) = 2` bits.</details>

---
← [Information (Surprise)](../1-surprise-and-entropy/01-information-surprise.md)  |  [Cross-Entropy](../2-comparing-distributions/03-cross-entropy.md) →


# Cross-Entropy

*📓 Notes · [06 · Information Theory](../README.md) · comparing distributions*

**📝 In simple words:** The average surprise when the truth follows `p` but you *predicted* `q`.

**📐 Formula to remember:** `H(p, q) = −Σ pᵢ log qᵢ`

**🔢 Small example:** true label is class 1 (`p = [1, 0]`). Model says `q = [0.7, 0.3]`.

**✏️ Let me calculate it:** `−(1·ln 0.7 + 0·ln 0.3) = −ln 0.7 = 0.357` nats. If the model said `0.99` → `0.010`. If it said `0.01` → `4.6`. Confident and wrong gets punished hard.

**🤖 Why AI uses it:** it's *the* classification loss, and the LLM training loss (next-token cross-entropy).

In [ ]:
p = np.array([1, 0]); q = np.array([0.7, 0.3])
print(-np.sum(p * np.log(q)))                 # 0.357
for c in (0.99, 0.5, 0.01): print(c, -np.log(c))

**🧪 Try it myself:** True class prob predicted as 0.5. Loss (nats)? <details><summary>Solution</summary>`−ln 0.5 = 0.693`.</details>

---
← [Entropy](../1-surprise-and-entropy/02-entropy.md)  |  [KL Divergence](../2-comparing-distributions/04-kl-divergence.md) →


# KL Divergence

*📓 Notes · [06 · Information Theory](../README.md) · comparing distributions*

**📝 In simple words:** The extra cost of using `q` when the truth is `p`; a "distance" between distributions (not symmetric).

**📐 Formula to remember:** `KL(p‖q) = Σ pᵢ log(pᵢ/qᵢ) = H(p, q) − H(p)`

**🔢 Small example:** `p = [0.5, 0.5]`, `q = [0.9, 0.1]`.

**✏️ Let me calculate it:** `H(p) = 1`, `H(p,q) = −0.5log₂0.9 − 0.5log₂0.1 = 0.076 + 1.661 = 1.737` → `KL = 0.737` bits.

**🤖 Why AI uses it:** minimizing cross-entropy = minimizing KL, since `H(p)` is fixed. KL also appears in VAEs, RLHF (keeping a fine-tuned model near the original), and distillation.

In [ ]:
p = np.array([0.5, 0.5]); q = np.array([0.9, 0.1])
print(np.sum(p * np.log2(p / q)))             # 0.737

**🧪 Try it myself:** `KL(p‖p)`? <details><summary>Solution</summary>0. Identical distributions.</details>

---
← [Cross-Entropy](../2-comparing-distributions/03-cross-entropy.md)  |  [Mutual Information](../3-mutual-info-and-why/05-mutual-information.md) →


# Mutual Information

*📓 Notes · [06 · Information Theory](../README.md) · mutual info and why*

**📝 In simple words:** How much knowing one variable reduces uncertainty about another.

**📐 Formula to remember:** `I(X;Y) = Σ P(x,y) log[P(x,y) / (P(x)P(y))]`

**🔢 Small example:** joint `[[0.4, 0.1],[0.1, 0.4]]`. Marginals are 0.5 each.

**✏️ Let me calculate it:** `2·0.4·log₂(0.4/0.25) + 2·0.1·log₂(0.1/0.25) = 0.543 − 0.264 = 0.278` bits. Independent variables would give 0.

**🤖 Why AI uses it:** feature selection, representation learning (InfoNCE / contrastive losses).

In [ ]:
J = np.array([[0.4, 0.1], [0.1, 0.4]])
px, py = J.sum(1), J.sum(0)
print(np.sum(J * np.log2(J / np.outer(px, py))))   # 0.278

**🧪 Try it myself:** MI when `P(x,y) = P(x)P(y)`? <details><summary>Solution</summary>0 (`log 1 = 0`).</details>

---
← [KL Divergence](../2-comparing-distributions/04-kl-divergence.md)  |  [Why This Shows Up in ML and LLMs](../3-mutual-info-and-why/06-why-this-shows-up-in-ml-and-llms.md) →


# Why This Shows Up in ML and LLMs

*📓 Notes · [06 · Information Theory](../README.md) · mutual info and why*

```text
 Entropy H(p)          = unavoidable uncertainty of the data
 Cross-entropy H(p,q)  = what the model actually pays
 KL(p‖q)               = H(p,q) − H(p)  = the model's avoidable mistake
```

- Training an LLM minimizes next-token cross-entropy → makes `q` close to the true text distribution.
- **Perplexity** `= e^(cross-entropy)`: "how many choices the model is effectively torn between" (chapter 10).
- Temperature/sampling change the entropy of the output distribution.

---
← [Mutual Information](../3-mutual-info-and-why/05-mutual-information.md)